In [ ]:
import torch.nn as nn
from torch.nn import Conv2d as cnv
from torch.nn import BatchNorm2d as BN
from torch.nn import PReLU as prelu
from torch.nn import PixelShuffle as ps
from torch.nn import LeakyReLU as lrelu
from torch.nn import Linear as lin
from torch.nn import Sigmoid as sig
from torch.nn import Sequential as seq
import torch.nn.functional as F
import torch;
from IPython.display import display

import re;
from torchvision import transforms;
from torchvision import datasets;
import torchvision;
from PIL import Image;
from torch.utils.data import Dataset, DataLoader;
import tqdm;
import numpy as np;
from torch.utils.checkpoint import checkpoint;

In [ ]:
# this is the module where we get out dataset
import urllib.request, zipfile, io

def download_and_unzip(url, extract_to='.'):
    # Send an HTTP GET request, download the file, and save it in a BytesIO object
    with urllib.request.urlopen(url) as response:
        file_content = response.read()
        zip_file_like = io.BytesIO(file_content)

        # Unzip the file
        with zipfile.ZipFile(zip_file_like) as zip_file:
            zip_file.extractall(extract_to)

download_and_unzip(url='http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip', extract_to='./lmao/valid')
download_and_unzip(url='http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip', extract_to='./lmao')

transformTrain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.CenterCrop(200)
]);

transformTest = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.CenterCrop(800)
]);

transformTrainFull = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
]);

transformTestFull = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
]);

train_dataset = datasets.ImageFolder(root='./data/DIV2K_train_LR_bicubic', transform=transformTrain);
label_dataset = datasets.ImageFolder(root='./data/train', transform=transformTest);

train_datasetFull = datasets.ImageFolder(root='./lmao/DIV2K_valid_LR_bicubic', transform=transformTrainFull);
label_datasetFull = datasets.ImageFolder(root='./lmao/valid', transform=transformTestFull);

class IMGDataset(Dataset):
  def __init__(self, train, label):
    self.labels = label;
    self.images = train;

  def __len__(self):
    return len(self.images);

  def __getitem__(self, idx):
    return self.images[idx][0], self.labels[idx][0];

IMGData = IMGDataset(train_dataset, label_dataset);
IMGDataFull = IMGDataset(train_datasetFull, label_datasetFull);
print(IMGDataFull.images, IMGDataFull.labels, IMGData.images, IMGData.labels)

In [ ]:
#our generator class
class generator_model(nn.Module):
  def __init__(self):
     super(generator_model, self).__init__();

     self.preProcess = nn.Sequential(
         nn.Conv2d(in_channels=3, out_channels=64, kernel_size=9, stride=1, padding=4),
         nn.PReLU(num_parameters=64)
     );

    #the b blocks dont have the final elementwise Sum layer, as to continue with skip connections
     self.BresBlock1 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock2 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock3 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock4 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock5 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock6 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock7 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock8 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock9 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock10 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock11 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock12 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock13 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock14 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock15 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.BresBlock16 = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     self.finBlock = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
         nn.BatchNorm2d(num_features=64)
     );

     #skip connection

     self.preOutput = nn.Sequential(
         nn.Conv2d(in_channels=64, out_channels=256, kernel_size=3, stride=1, padding=1),
         nn.PixelShuffle(upscale_factor=2),
         nn.PReLU(num_parameters=64),
         nn.Conv2d(in_channels=64, out_channels=256, kernel_size=3, stride=1, padding=1),
         nn.PixelShuffle(upscale_factor=2),
         nn.PReLU(num_parameters=64)
     )

     self.output = nn.Conv2d(in_channels=64, out_channels=3, kernel_size=9, stride=1, padding=4);



  def forward(self, x):
    skip_main = self.preProcess(x);
    skip = skip_main;

    skip = skip + self.BresBlock1(skip);
    skip = skip + self.BresBlock2(skip);
    skip = skip + self.BresBlock3(skip);
    skip = skip + self.BresBlock4(skip);
    skip = skip + self.BresBlock5(skip);
    skip = skip + self.BresBlock6(skip);
    skip = skip + self.BresBlock7(skip);
    skip = skip + self.BresBlock8(skip);
    skip = skip + self.BresBlock9(skip);
    skip = skip + self.BresBlock10(skip);
    skip = skip + self.BresBlock11(skip);
    skip = skip + self.BresBlock12(skip);
    skip = skip + self.BresBlock13(skip);
    skip = skip + self.BresBlock14(skip);
    skip = skip + self.BresBlock15(skip);
    x = skip + self.BresBlock16(skip);

    x = skip_main + self.finBlock(x);

    x = self.preOutput(x);
    return self.output(x);

  def initialize_weights(self):
      for m in self.modules():
        if (isinstance(m, nn.Conv2d)):
          nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
          if m.bias is not None:
            nn.init.constant_(m.bias, 0);
          elif (isinstance(m, nn.BatchNorm2d)):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

In [ ]:
#discriminator
#k = kernel size, n = batch size, s = stride
def __discriminator_model():

    model = seq(
        #k3n64s1
        cnv(3, 64, 3, 1),
        lrelu(0.2),
        #k3n64s2 -> k3n128s1
        cnv(64, 64, 3, 2),
        BN(64),
        lrelu(0.2),
    )

    res_units = [64, 128, 256, 512, 1024]
    kernel_size = 3
    stride_units = [1, 2]
    learning_rate = pow(10, -4)
    #layers k2n64s2 k3n128s1 k3n128s2 k3n256s1 k3n256s2 k3n512s1 k3n512s2
    for i in range(len(res_units)-2):
      block = seq(
          #in channel is past layer n val output is current layer n val
          #k3n128s1, k3n256s1, k3n512s1
          cnv(res_units[i], res_units[i+1], kernel_size, stride_units[0]),
          BN(res_units[i+1]),
          lrelu(0.2),
          #k3n128s2, k3n256s2, k3n512s2
          cnv(res_units[i+1], res_units[i+1], kernel_size, stride_units[1]),
          BN(res_units[i+1]),
          lrelu(0.2)
      )
      model.add_module('block group '+str(i+2), block)

    model.add_module('avgpool', nn.AdaptiveAvgPool2d((6, 6)))
    model.add_module('flatten', nn.Flatten())
    model.add_module('fc1', lin(512 * 6 * 6, 1024))
    model.add_module('leaky_fc', lrelu(0.2))
    model.add_module('fc2', lin(1024, 1))
    model.add_module('sigmoid', sig())

    return model;

In [ ]:
class VGGLoss(nn.Module):
    def __init__(self, device):
        super(VGGLoss, self).__init__()
        vgg19 = torchvision.models.vgg19(pretrained=True)
        self.layers = nn.Sequential(*list(vgg19.features.children())[:29])
        self.layers.eval()
        for param in self.layers.parameters():
            param.requires_grad = False
        self.loss = nn.MSELoss()
        self.device = device
        self.to(device)

    def forward(self, x, y):
        x_feat = self.layers(x)
        with torch.no_grad():  # only for the VGG layers
            y_feat = self.layers(y)
        return self.loss(x_feat, y_feat)


In [ ]:
#training func
import gc, json

def trainFirst(generator, train_loader):
  loss_function = nn.MSELoss();
  opt = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.9, 0.999));
  finalLoss = [];

  for epoch in range(1900):
    losses = [];
    for lr_imgs, hr_imgs in tqdm.tqdm(train_loader):
      lr_imgs = lr_imgs.to(device)
      hr_imgs = hr_imgs.to(device)

      sr_imgs = generator(lr_imgs)
      loss = loss_function(sr_imgs, hr_imgs)
      losses.append(loss.detach().item())
      loss.backward();
      opt.step();
      opt.zero_grad();
      del lr_imgs, hr_imgs, sr_imgs, loss;
      torch.cuda.empty_cache(); gc.collect();

    print("Loss for epoch", epoch, "is", np.mean(losses))
    finalLoss.append(np.mean(losses));

  return finalLoss;

def trainSecond(generator, train_loader):
  loss_function = nn.MSELoss();
  opt = torch.optim.Adam(generator.parameters(), lr=0.00005, betas=(0.9, 0.999));
  i = 0;
  finalLoss = [];
  for epoch in range(200):
    losses = [];
    for lr_imgs, hr_imgs in tqdm.tqdm(train_loader):
      i+=1;
      lr_imgs = lr_imgs.to(device)
      hr_imgs = hr_imgs.to(device)

      sr_imgs = generator(lr_imgs)
      loss = loss_function(sr_imgs, hr_imgs)
      losses.append(loss.detach().item())
      loss.backward();

      if (i % 10 == 0):
        opt.step();
        opt.zero_grad();
        i = 0;

      del lr_imgs, hr_imgs, sr_imgs, loss;
      torch.cuda.empty_cache(); gc.collect();

    print("Loss for epoch", epoch, "is", np.mean(losses))
    finalLoss.append(np.mean(losses))

  return finalLoss;

train_loader = DataLoader(IMGData, batch_size=10, shuffle=True);
train_loaderFull = DataLoader(IMGDataFull, batch_size=1, shuffle=True);

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = generator_model().to(device);
model.initialize_weights();

before2K = trainFirst(generator=model, train_loader=train_loader);
torch.save(model.state_dict(), 'model_weightsAfter2000.pth')
after2K = trainSecond(generator=model, train_loader=train_loaderFull);
torch.save(model.state_dict(), 'model_weightsAfter2100.pth')

In [ ]:
class BilinearUpscaler(nn.Module):
    def __init__(self, upscale_factor):
        super().__init__()
        if not isinstance(upscale_factor, int) or upscale_factor <= 0:
            raise ValueError("upscale_factor must be a positive integer.")
        self.upscale_factor = upscale_factor

    def forward(self, x):
        output = F.interpolate(x,
                               scale_factor=self.upscale_factor,
                               mode='bilinear',
                               align_corners=False)
        return output

In [ ]:
#display(transforms.functional.to_pil_image(y.to(torch.uint8)))
import math;
mean = torch.tensor([0.485, 0.456, 0.406]).to(device);
std = torch.tensor([0.229, 0.224, 0.225]).to(device);
#model = generator_model();
model = BilinearUpscaler(upscale_factor=4);
#model.load_state_dict(torch.load('srgan_gen_weights.pth', weights_only=True));
#model.load_state_dict(torch.load('model_weightsAfter2100.pth', weights_only=True));
model = model.to(device)
xcv = [];
l = nn.MSELoss();

torch.cuda.empty_cache(); gc.collect();

for i in range(100):
    x, y = IMGDataFull.__getitem__(i)
    xcv.append(10 * math.log10((255*255)/l((((model(x.unsqueeze(0).to(device)).squeeze(0)) * std.view(3, 1, 1) + mean.view(3, 1, 1))*255.0), ((y.to(device) * std.view(3, 1, 1) + mean.view(3, 1, 1))*255.0)).detach().item()));
    print(i, " ", xcv[i])

print(np.mean(xcv), "and", np.max(xcv))

In [ ]:
#for i in range(100):
x, y = IMGDataFull.__getitem__(1);
y = y.to(device);
asd = model(x.unsqueeze(0).to(device)).squeeze(0);
asd = (asd*std.view(3, 1, 1) + mean.view(3, 1, 1))*255;
asd = transforms.functional.to_pil_image(asd.to(torch.uint8));
#asd.save(f"./BiImg/0{i+1+800}.png");
display(asd)
display(transforms.functional.to_pil_image(((y*std.view(3, 1, 1) + mean.view(3, 1, 1))*255).to(torch.uint8)))